# RG374 FACS analysis IFNAR fl LysMcre CpG

In [ ]:
options(warn=-1)

In [ ]:
library_load <- suppressMessages(
    
    suppressWarnings(
        
        list(

            library(stats), 
            library(emmeans), 
            library(outliers), # Grubbs outlier detection 
            
            # Data 
            library(tidyverse), 
            library(data.table), 
            library(reactable), 

            # Plotting 
            library(ComplexHeatmap), 
            library(patchwork), 
            library(cowplot), 
            library(ggrepel)

        )
    )
)

In [ ]:
random_seed <- 42
set.seed(random_seed)

In [ ]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20200109SPLENO")

In [ ]:
# Plotting Theme
source("plotting_global.R")
ggplot2::theme_set(theme_global_set(size_select=1)) # From project global source()

# Parameter 

In [ ]:
color$sample_group<- c("D0 +/+"="#66c2a5", "D0 cre/+"="#00634A", "D1 +/+"="#cd34b5", "D1 cre/+"="#FFAC1E", "D3 +/+"="#cd34b5", "D3 cre/+"="#FFAC1E", "D6 +/+"="#cd34b5", "D6 cre/+"="#FFAC1E")

# Import data 

In [128]:
data <- rbind(
    
    read.csv("data/RG401/p1.csv"), 
    read.csv("data/RG401/p2.csv")

) %>% dplyr::mutate(genotype=gsub("'", "", genotype), dpi=ifelse(dpi=="Ctl", "D0", dpi), sample_group=paste(dpi, genotype))
dim(data)

[1] 510   9

In [129]:
data <- data %>%
    group_by(gene) %>%
    mutate(
        
        Q1 = quantile(dCT, 0.25, na.rm=TRUE),
        Q3 = quantile(dCT, 0.75, na.rm=TRUE),
        IQR = Q3 - Q1,
        outlier = dCT < (Q1 - 1.5*IQR) | dCT > (Q3 + 1.5*IQR)
    
    ) %>% dplyr::filter(!outlier)
dim(data)

[1] 490  13

In [130]:
data <- data %>% dplyr::group_by(gene, dpi, genotype) %>% 
    dplyr::summarise(
        
        dCT_mean=mean(dCT), 
        dCT_sd=sd(dCT), 
        n=n(), 
        .groups="drop"
    
    ) %>% 
    
    tidyr::pivot_wider(names_from=genotype, values_from=c(dCT_mean, dCT_sd, n)) %>% 

    dplyr::mutate(
        
        ddCT_mean=`dCT_mean_cre/+`-`dCT_mean_+/+`, 
        ddCT_se=sqrt((`dCT_sd_+/+` / sqrt(`n_+/+`))^2 + (`dCT_sd_cre/+` / sqrt(`n_cre/+`))^2), 
        dCT_se_min=ddCT_mean-ddCT_se, 
        dCT_se_max=ddCT_mean+ddCT_se
        
    
    
    ) %>% 

    dplyr::mutate(sample_group=ifelse(sign(ddCT_mean)>0, paste(dpi, "cre/+"), paste(dpi, "+/+")))

In [136]:
p <- lapply(split(data, f=data$gene), function(x) {

    y_limit <- max(c(abs(x$dCT_se_min), abs(x$dCT_se_max)))
    
    ggplot(x, aes(x=dpi, y=ddCT_mean, fill=sample_group)) + 
        ggtitle(x$gene[1]) + 
        geom_hline(yintercept = 0) + 
        geom_bar(stat="identity", color="black", size=0.1, width=0.8) + 
        geom_errorbar(aes(x=dpi, ymin=dCT_se_min, ymax=dCT_se_max), width=0.4, colour="black", size=0.25) + 
        scale_y_continuous(limits=c(-y_limit, +y_limit)) + 
        scale_fill_manual(values=color$sample_group) + 
        facet_grid(~dpi, scales="free") + 
        theme(legend.position = "none") + theme_global_set(4)
    
}
      )

p <- lapply(p, function(p) egg::set_panel_size(p, width=unit(0.5, "cm"), height=unit(2.0, "cm")))
p <- do.call(gridExtra::arrangeGrob, c(p, ncol=4, nrow=ceiling(length(p)/4)))

In [137]:
pdf("result/figures/4_RG401/qpcr.pdf", width=7.5, height=1.9*ceiling(length(p)/5))

grid::grid.draw(p)

dev.off()

pdf 
  2